# Zebrafish Gene–Gene (STRING) — Relation-Wise KG Triple Construction

## Purpose

This notebook processes **Protein–Protein interaction data** from the STRING database for Zebrafish (*Danio rerio*) and transforms it into standardized Gene–Gene relation-wise Knowledge Graph (KG) triples. Ensembl protein IDs are mapped to gene symbols via gProfiler.

## Pipeline Overview

| Step | Section | Description |
|------|---------|-------------|
| 1 | Setup & gProfiler | Load gProfiler mapping (Ensembl Protein ID → Gene Symbol + Description) |
| 2 | STRING Raw Preprocessing | Load raw STRING TSV, clean protein IDs, add metadata, save CSV |
| 3 | KG Triple Construction | Load CSV, map proteins to genes, add metadata, annotate descriptions |
| 4 | Filter, Validate, Deduplicate & Export | Remove unmapped entries, validate, deduplicate, export final CSV |

## Final Output Schema

| Column | Description |
|--------|-------------|
| `head` | Gene symbol (head gene) |
| `relation` | `Gene_Gene` |
| `tail` | Gene symbol (tail gene) |
| `head_type` | `Gene` |
| `relation_type` | (NaN — no sub-relation) |
| `tail_type` | `Gene` |
| `kg_source` | `STRING` |
| `head_id_is` | `NCBI_ID` |
| `tail_id_is` | `NCBI_ID` |
| `head_detail_name` | Gene description from gProfiler |
| `tail_detail_name` | Gene description from gProfiler |
| `species` | `D.rerio` |

## Data Download Instructions

All databases used in this notebook must be downloaded prior to execution.
Please refer to the **central download instructions document** for detailed steps:

📄 **[Download Instructions — Link to be added]**

### Required Files

| File | Source | Description |
|------|--------|-------------|
| `7955.protein.links.v12.0.txt` | STRING | Raw protein–protein interaction file for zebrafish (tax_id: 7955) |
| `gProfiler_drerio_*.csv` | gProfiler | Ensembl protein ID to gene symbol/description mapping |

---
## Setup

Import required libraries and define all base paths.

In [4]:
import pandas as pd
import numpy as np

In [5]:
BASE_PATH = '/storage/Arushi/090526_EvoAge/kg_formation/data_collection/'

In [11]:

# gProfiler mapping file (Ensembl protein ID → gene symbol)
GPROFILER_PATH = f'{BASE_PATH}stitch/drerio/gProfiler_drerio_4-9-2025_5-19-17 PM.csv'

# Raw STRING protein-protein interaction file for zebrafish
STRING_RAW_PATH = f'{BASE_PATH}string/drer/7955.protein.links.v12.0.txt'

# Intermediate CSV output
# STRING_CSV_PATH = f'{BASE_PATH}/ZEBRAFISH/PROTEIN_PROTEIN/ZF_P_P_STRING_Triples.csv'

# Final output path
OUTPUT_PATH = f'/storage/Arushi/090526_EvoAge/kg_formation/processed_data/string/drer/string_Zebra_GENE_GENE.csv'

---
## 1. Load gProfiler Mapping (Ensembl Protein ID → Gene Symbol + Description)

gProfiler provides a mapping from Ensembl protein IDs (e.g., `ENSDARP00000000004`) to gene symbols (e.g., `slc35a5`) and descriptions. Both head and tail proteins will be mapped to gene symbols using this dictionary.

In [7]:
# =============================================================================
# Load gProfiler data and build mapping dictionaries
# =============================================================================

gprofiler_mapping = pd.read_csv(GPROFILER_PATH)
gprofiler_mapping = gprofiler_mapping[~gprofiler_mapping['converted_alias'].isna()]

# Dictionary: Ensembl Protein ID → Gene Symbol
protein_to_gene_dict = dict(zip(gprofiler_mapping['initial_alias'], gprofiler_mapping['name']))

# Dictionary: Gene Symbol → Gene Description (used for head/tail_detail_name)
gene_to_description_dict = dict(zip(gprofiler_mapping['name'], gprofiler_mapping['description']))

print(f"gProfiler entries loaded: {len(gprofiler_mapping):,}")
print(f"Protein → Gene dictionary size: {len(protein_to_gene_dict):,}")
print(f"Gene → Description dictionary size: {len(gene_to_description_dict):,}")
gprofiler_mapping.head()

gProfiler entries loaded: 44,669
Protein → Gene dictionary size: 24,474
Gene → Description dictionary size: 24,117


,initial_alias,converted_alias,name,description,namespace
0,ENSDARP00000000004,ENSDARP00000000004,slc35a5,solute carrier family 35 member A5 [Source:NCB...,ENSP
1,ENSDARP00000000005,ENSDARP00000000005,ccdc80,coiled-coil domain containing 80 [Source:NCBI ...,ENSP
2,ENSDARP00000000042,ENSDARP00000000042,mcm6l,"MCM6 minichromosome maintenance deficient 6, l...",ENSP
3,ENSDARP00000000042,ENSDARP00000119510,mcm6l,"MCM6 minichromosome maintenance deficient 6, l...",ENSP
4,ENSDARP00000000069,ENSDARP00000000069,nherf1a,NHERF family PDZ scaffold protein 1a [Source:Z...,ENSP


---
## 2. STRING Raw Preprocessing (TXT → CSV)

This section loads the raw STRING protein–protein interaction file, cleans protein identifiers, adds relation metadata, and saves as CSV.

**Processing steps:**
1. Load raw TXT (columns: protein1, protein2, combined_score)
2. Strip species prefix (`7955.`) from both protein columns
3. Rename columns to Head/Tail
4. Add relation metadata (Relation, Relation_Source, head_type, tail_type)
5. Remove trailing version numbers from protein IDs
6. Save as CSV

**Note:** This section only needs to be run once. After the CSV is saved, Section 3 can load it directly.

In [13]:
# =============================================================================
# Load raw STRING protein-protein interaction file
# =============================================================================
# The file uses whitespace as separator
# Columns: protein1, protein2, combined_score

string_ppi_raw = pd.read_csv(STRING_RAW_PATH, sep='\\s')

print(f"Raw STRING interactions loaded: {len(string_ppi_raw):,}")
print(f"Columns: {list(string_ppi_raw.columns)}")
string_ppi_raw.head()

/tmp/ipykernel_1663121/1454153917.py:7: ParserWarning: Falling back to the 'python' engine because the 'c' engine does not support regex separators (separators > 1 char and different from '\s+' are interpreted as regex); you can avoid this warning by specifying engine='python'.
  string_ppi_raw = pd.read_csv(STRING_RAW_PATH, sep='\\s')


Raw STRING interactions loaded: 13,815,202
Columns: ['protein1', 'protein2', 'combined_score']


,protein1,protein2,combined_score
0,7955.ENSDARP00000000004,7955.ENSDARP00000153477,152
1,7955.ENSDARP00000000004,7955.ENSDARP00000073803,186
2,7955.ENSDARP00000000004,7955.ENSDARP00000108347,150
3,7955.ENSDARP00000000004,7955.ENSDARP00000115664,171
4,7955.ENSDARP00000000004,7955.ENSDARP00000102719,151


In [14]:
# =============================================================================
# Clean protein identifiers and add relation metadata
# =============================================================================

# Strip species prefix '7955.' from both protein columns
string_ppi_raw['protein1'] = string_ppi_raw['protein1'].str.replace('7955.', '')
string_ppi_raw['protein2'] = string_ppi_raw['protein2'].str.replace('7955.', '')

# Rename to Head/Tail
string_ppi_raw = string_ppi_raw.rename(columns={'protein1': 'Head', 'protein2': 'Tail'})

# Add relation and entity type metadata
string_ppi_raw['Relation'] = 'Protein_Protein'
string_ppi_raw['Relation_Source'] = 'STRING'
string_ppi_raw['head_type'] = 'Protein'
string_ppi_raw['tail_type'] = 'Protein'

# Reorder columns
string_ppi_raw = string_ppi_raw[['Head', 'Relation', 'Tail', 'combined_score', 'head_type', 'tail_type', 'Relation_Source']]

# Remove trailing version numbers (e.g., ENSDARP00000068614.1 → ENSDARP00000068614)
string_ppi_raw['Head'] = string_ppi_raw['Head'].apply(lambda x: x.rsplit('.', 1)[0] if x.count('.') > 1 else x)
string_ppi_raw['Tail'] = string_ppi_raw['Tail'].apply(lambda x: x.rsplit('.', 1)[0] if x.count('.') > 1 else x)

print(f"Protein IDs cleaned")
string_ppi_raw.head()

Protein IDs cleaned


,Head,Relation,Tail,combined_score,head_type,tail_type,Relation_Source
0,ENSDARP00000000004,Protein_Protein,ENSDARP00000153477,152,Protein,Protein,STRING
1,ENSDARP00000000004,Protein_Protein,ENSDARP00000073803,186,Protein,Protein,STRING
2,ENSDARP00000000004,Protein_Protein,ENSDARP00000108347,150,Protein,Protein,STRING
3,ENSDARP00000000004,Protein_Protein,ENSDARP00000115664,171,Protein,Protein,STRING
4,ENSDARP00000000004,Protein_Protein,ENSDARP00000102719,151,Protein,Protein,STRING


In [19]:
protein_to_gene_dict

{'ENSDARP00000000004': 'slc35a5',
 'ENSDARP00000000005': 'ccdc80',
 'ENSDARP00000000042': 'mcm6l',
 'ENSDARP00000000069': 'nherf1a',
 'ENSDARP00000000070': 'dap',
 'ENSDARP00000000160': 'thraa',
 'ENSDARP00000000221': 'krt91',
 'ENSDARP00000000250': 'slc40a1',
 'ENSDARP00000000280': 'stat1b',
 'ENSDARP00000000382': 'TRIO',
 'ENSDARP00000000392': 'pde6a',
 'ENSDARP00000000488': 'TNNI1',
 'ENSDARP00000000748': 'mgat4b',
 'ENSDARP00000000803': 'actn3b',
 'ENSDARP00000000854': 'ptpn4a',
 'ENSDARP00000000952': 'ofd1',
 'ENSDARP00000000956': 'fkbp8',
 'ENSDARP00000000979': 'opn1mw4',
 'ENSDARP00000000985': 'slc16a3b',
 'ENSDARP00000001039': 'pcgf1',
 'ENSDARP00000001137': 'sp3a',
 'ENSDARP00000001151': 'gtf2e1',
 'ENSDARP00000001156': 'osgepl1',
 'ENSDARP00000001158': 'opn1mw1',
 'ENSDARP00000001187': 'adam8a',
 'ENSDARP00000001220': 'psmb9a',
 'ENSDARP00000001240': 'tomm34',
 'ENSDARP00000001250': 'pms1',
 'ENSDARP00000001256': 'psmb13a',
 'ENSDARP00000001281': 'CACNA2D3',
 'ENSDARP00000001

---
## 3. KG Triple Construction (CSV → Annotated Triples)



In [16]:
gene_gene_df = string_ppi_raw

print(f"STRING Protein-Protein interactions loaded: {len(gene_gene_df):,}")

STRING Protein-Protein interactions loaded: 13,815,202


In [18]:
# =============================================================================
# Map protein IDs to gene symbols and add KG metadata
# =============================================================================

# Add KG metadata
gene_gene_df['relation_type'] = np.nan
gene_gene_df['kg_source'] = 'STRING'

# Map Ensembl Protein IDs to gene symbols using gProfiler (both head and tail)
gene_gene_df['tail'] = gene_gene_df['Tail'].map(protein_to_gene_dict)
gene_gene_df['head'] = gene_gene_df['Head'].map(protein_to_gene_dict)

# Set identifier namespace
gene_gene_df['head_id_is'] = 'NCBI_ID'
gene_gene_df['tail_id_is'] = 'NCBI_ID'

print(f"Protein IDs mapped to gene symbols")
print(f"Head mapped: {gene_gene_df['head'].notna().sum():,}")
print(f"Tail mapped: {gene_gene_df['tail'].notna().sum():,}")

Protein IDs mapped to gene symbols
Head mapped: 13,815,202
Tail mapped: 13,815,202


In [20]:
# =============================================================================
# Map gene descriptions for both head and tail
# =============================================================================

gene_gene_df['tail_detail_name'] = gene_gene_df['tail'].map(gene_to_description_dict)
gene_gene_df['head_detail_name'] = gene_gene_df['head'].map(gene_to_description_dict)

print(f"Descriptions mapped")
print(f"Head descriptions: {gene_gene_df['head_detail_name'].notna().sum():,}")
print(f"Tail descriptions: {gene_gene_df['tail_detail_name'].notna().sum():,}")

Descriptions mapped
Head descriptions: 11,686,152
Tail descriptions: 11,686,152


---
## 4. Filter, Validate, Deduplicate & Export

Filter out entries with unmapped genes/descriptions, add final metadata, validate data quality, deduplicate, and export.

In [21]:
# =============================================================================
# Filter out entries with unmapped descriptions
# =============================================================================

before_count = len(gene_gene_df)
gene_gene_df = gene_gene_df[~gene_gene_df['tail_detail_name'].isna()]
gene_gene_df = gene_gene_df[~gene_gene_df['head_detail_name'].isna()]
after_count = len(gene_gene_df)

print(f"Before filtering: {before_count:,}")
print(f"After filtering: {after_count:,}")
print(f"Removed: {before_count - after_count:,} triples with unmapped genes")

Before filtering: 13,815,202
After filtering: 9,925,668
Removed: 3,889,534 triples with unmapped genes


In [22]:
# =============================================================================
# Add final metadata
# =============================================================================

gene_gene_df['species'] = 'D.rerio'
gene_gene_df['relation'] = 'Gene_Gene'
gene_gene_df['head_type'] = 'Gene'
gene_gene_df['tail_type'] = 'Gene'

print(f"Final metadata added. Triples: {len(gene_gene_df):,}")

Final metadata added. Triples: 9,925,668


In [23]:
# =============================================================================
# Data quality validation
# =============================================================================

true_nan_count = gene_gene_df.isna().sum()
string_nan_count = gene_gene_df.apply(lambda col: col.astype(str).str.upper().eq('NAN').sum())

nan_summary = pd.DataFrame({
    'NaN_count': true_nan_count,
    "'NAN'_string_count": string_nan_count,
    'Total_NaN_like': true_nan_count + string_nan_count
})

print("NaN validation summary:")
nan_summary

NaN validation summary:


,NaN_count,'NAN'_string_count,Total_NaN_like
Head,0,0,0
Relation,0,0,0
Tail,0,0,0
combined_score,0,0,0
head_type,0,0,0
tail_type,0,0,0
Relation_Source,0,0,0
relation_type,9925668,9925668,19851336
kg_source,0,0,0
tail,0,0,0


In [24]:
# =============================================================================
# Deduplicate by grouping on (head, relation, tail)
# =============================================================================

group_cols = ['head', 'relation', 'tail']

def merge_sources(x):
    """Merge multiple kg_source values with '::' separator."""
    return '::'.join(sorted(set(x.dropna())))

gene_gene_dedup = gene_gene_df.groupby(group_cols, as_index=False).agg({
    'head_type': 'first',
    'relation_type': 'first',
    'tail_type': 'first',
    'kg_source': merge_sources,
    'head_id_is': 'first',
    'tail_id_is': 'first',
    'head_detail_name': 'first',
    'tail_detail_name': 'first',
    'species': 'first'
})

print(f"Triples before deduplication: {len(gene_gene_df):,}")
print(f"Triples after deduplication: {len(gene_gene_dedup):,}")
print(f"Duplicates removed: {len(gene_gene_df) - len(gene_gene_dedup):,}")
gene_gene_dedup.head()

Triples before deduplication: 9,925,668
Triples after deduplication: 9,691,758
Duplicates removed: 233,910


,head,relation,tail,head_type,relation_type,tail_type,kg_source,head_id_is,tail_id_is,head_detail_name,tail_detail_name,species
0,42sp43,Gene_Gene,ARID3A,Gene,NaN,Gene,STRING,NCBI_ID,NCBI_ID,P43 5S RNA-binding protein [Source:ZFIN;Acc:ZD...,AT-rich interaction domain 3A [Source:HGNC Sym...,D.rerio
1,42sp43,Gene_Gene,CDY2B,Gene,NaN,Gene,STRING,NCBI_ID,NCBI_ID,P43 5S RNA-binding protein [Source:ZFIN;Acc:ZD...,chromodomain Y-linked 2B [Source:HGNC Symbol;A...,D.rerio
2,42sp43,Gene_Gene,DEPDC7,Gene,NaN,Gene,STRING,NCBI_ID,NCBI_ID,P43 5S RNA-binding protein [Source:ZFIN;Acc:ZD...,DEP domain containing 7 [Source:HGNC Symbol;Ac...,D.rerio
3,42sp43,Gene_Gene,FAM135B,Gene,NaN,Gene,STRING,NCBI_ID,NCBI_ID,P43 5S RNA-binding protein [Source:ZFIN;Acc:ZD...,family with sequence similarity 135 member B [...,D.rerio
4,42sp43,Gene_Gene,GADD45GIP1,Gene,NaN,Gene,STRING,NCBI_ID,NCBI_ID,P43 5S RNA-binding protein [Source:ZFIN;Acc:ZD...,GADD45G interacting protein 1 [Source:HGNC Sym...,D.rerio


In [26]:
OUTPUT_PATH

'/storage/Arushi/090526_EvoAge/kg_formation/processed_data/string/drer/string_Zebra_GENE_GENE.csv'

In [27]:
# =============================================================================
# Export final deduplicated Gene-Gene KG triples
# =============================================================================

gene_gene_dedup.to_csv(OUTPUT_PATH, index=None)
print(f"Final output saved to: {OUTPUT_PATH}")
print(f"Total triples exported: {len(gene_gene_dedup):,}")

Final output saved to: /storage/Arushi/090526_EvoAge/kg_formation/processed_data/string/drer/string_Zebra_GENE_GENE.csv
Total triples exported: 9,691,758


---
## Summary

This notebook processed STRING Protein–Protein interaction data for Zebrafish into standardized Gene–Gene KG triples:

1. **gProfiler setup** — Loaded ~29K Ensembl protein → gene symbol mappings
2. **Raw STRING preprocessing** — Loaded ~13.8M raw interactions, cleaned protein IDs, saved CSV
3. **KG triple construction** — Mapped both head and tail proteins to gene symbols via gProfiler
4. **Annotation & filtering** — Added gene descriptions, filtered unmapped entries (~13.8M → ~9.9M)
5. **Deduplication** — Grouped by (head, relation, tail) for unique final triples

### Output Files

| File | Description |
|------|-------------|

| `string_Zebra_GENE_GENE.csv` | Final: deduplicated relation-wise KG triples |